In [ ]:
import pandas as pd
import xlrd
import openpyxl
from pandas import ExcelWriter
from datetime import datetime
import os
from shutil import copyfile
from openpyxl.utils.cell import get_column_letter
import urllib.request, json
import geopoll_functions
import geopoll_modified
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
import re

In [2]:
# --- Basic Survey Metadata ---
template_version = "20250708"
iso3 = "MMR"
round_number = "13"
admin_level = "Admin 2"
user_name = "Edoardo"
language = "EN"

In [1]:
import os
import glob
import polars as pl

# --- 1. Target File to Validate ---
# This is the "Actual" file you just received from the country team
Q_file_path = r"C:\Temp\MMR_R13 household_questionnaire_t1_geopoll_EN_template_20250708_ISO3.xlsx"

# --- 2. Baseline Configuration ---
Q_baseline_option = "Template" # Options: "Template" or "Custom"
Q_custom_path = r""            # Only used if option is "Custom"

# --- 3. Master Repository Path ---
# This is where all your official FAO/GeoPoll templates live
TEMPLATE_REPO = r"C:\Users\edoar\Food and Agriculture Organization\OER - 01. DIEM Monitoring\03. Toolbox\02. Questionnaires"  

In [ ]:

import os
import glob

def get_latest_template(repo_path, enumerator, lang):
    """
    Finds the correct template based on provider (geopoll/kobo) and language.
    """
    # 1. Create a search pattern: e.g., "*geopoll_EN_template*.xlsx"
    # This ignores the version number and ISO3 tag, which change often.
    pattern = f"*_{enumerator.lower()}_{lang.upper()}_template*.xlsx"
    search_path = os.path.join(repo_path, pattern)
    
    # 2. Get all matching files
    matching_files = glob.glob(search_path)
    
    if not matching_files:
        raise FileNotFoundError(f"❌ No template found for {enumerator} in {lang} at {repo_path}")

    # 3. If multiple versions exist (e.g., 20250708 vs 2024...), pick the most recent one
    # Sorting by name usually works for date-coded versions, 
    # but sorting by modification time is safer.
    matching_files.sort(key=os.path.getmtime, reverse=True)
    
    return matching_files[0]

# --- Questionnaire Initialization ---
# We detect these from the file you want to validate
enumerator = "geopoll" # Returns "geopoll"
lang = language      # Returns "EN", "FR", etc.

# --- Step 1: Baseline Selection ---
execution_messages = ""
if Q_baseline_option == "Custom":
    Q_baseline = Q_custom_path
    baseline_source = "Custom File"
else:
    # This automatically finds the right file from your folder screenshot
    Q_baseline = get_latest_template(TEMPLATE_REPO, enumerator, lang)
    baseline_source = f"Master Template ({enumerator} {lang})"

execution_messages += f"\n- Baseline Source: {baseline_source}"
execution_messages += f"\n- Baseline File: {os.path.basename(Q_baseline)}"

print(execution_messages)


- Baseline Source: Master Template (geopoll EN)
- Baseline File: household_questionnaire_geopoll_EN_template_20250708_ISO3.xlsx


In [10]:
Q_baseline

'C:\\Users\\edoar\\Food and Agriculture Organization\\OER - 01. DIEM Monitoring\\03. Toolbox\\02. Questionnaires\\household_questionnaire_geopoll_EN_template_20250708_ISO3.xlsx'

### Step1 Configuration ###


In [25]:
from dataclasses import dataclass
from pathlib import Path
from typing import Literal, Optional

ReferenceMode = Literal["latest_template", "custom_reference", "previous_round"]
Enumerator = Literal["geopoll", "kobo"]
Language = Literal["EN", "FR", "ES", "AR", "PT"]

@dataclass
class ValidationConfig:
    template_repo: Path
    working_dir: Path
    questionnaire_file: str
    reference_mode: ReferenceMode
    enumerator: Enumerator
    language: Language
    custom_reference_file: Optional[str] = None
    previous_round_file: Optional[str] = None
    output_subfolder: str = "output"

cfg = ValidationConfig(
    template_repo=Path(r"C:\Users\edoar\Food and Agriculture Organization\OER - 01. DIEM Monitoring\03. Toolbox\02. Questionnaires"),
    working_dir=Path(r"C:\Questionnaire_Validation"),
    questionnaire_file="MMR_R13 household_questionnaire_t1_geopoll_EN_template_20250708_ISO3.xlsx",
    reference_mode="latest_template",
    enumerator="geopoll",
    language="EN",
    custom_reference_file=None,
    previous_round_file=None,
)

import re
from pathlib import Path

def extract_date_from_name(path: Path) -> int:
    m = re.search(r"template_(\d{8})", path.name)
    return int(m.group(1)) if m else 0

def find_latest_template(template_repo: Path, enumerator: str, language: str) -> Path:
    matches = []
    for path in template_repo.glob("*.xlsx"):
        name = path.name.lower()
        if enumerator.lower() not in name:
            continue
        if f"_{language.lower()}_" not in name:
            continue
        if "template" not in name:
            continue
        matches.append(path)

    if not matches:
        raise FileNotFoundError(
            f"No template found for enumerator={enumerator}, language={language} in {template_repo}"
        )

    matches.sort(key=lambda p: (extract_date_from_name(p), p.stat().st_mtime), reverse=True)
    return matches[0]

def resolve_reference_file(cfg: ValidationConfig) -> Path:
    if cfg.reference_mode == "latest_template":
        return find_latest_template(cfg.template_repo, cfg.enumerator, cfg.language)

    if cfg.reference_mode == "custom_reference":
        if not cfg.custom_reference_file:
            raise ValueError("custom_reference_file is required when reference_mode='custom_reference'")
        return cfg.working_dir / cfg.custom_reference_file

    if cfg.reference_mode == "previous_round":
        if not cfg.previous_round_file:
            raise ValueError("previous_round_file is required when reference_mode='previous_round'")
        return cfg.working_dir / cfg.previous_round_file

    raise ValueError(f"Unsupported reference_mode: {cfg.reference_mode}")

In [26]:
def prepare_run(cfg: ValidationConfig) -> dict:
    cfg.working_dir.mkdir(parents=True, exist_ok=True)
    output_dir = cfg.working_dir / cfg.output_subfolder
    output_dir.mkdir(parents=True, exist_ok=True)

    questionnaire_path = cfg.working_dir / cfg.questionnaire_file
    reference_path = resolve_reference_file(cfg)

    if not questionnaire_path.exists():
        raise FileNotFoundError(f"Questionnaire file not found: {questionnaire_path}")
    if not reference_path.exists():
        raise FileNotFoundError(f"Reference file not found: {reference_path}")

    result_file = output_dir / f"{questionnaire_path.stem.split()[0]}_{cfg.enumerator}_validated.xlsx"

    run_info = {
        "questionnaire_path": questionnaire_path,
        "reference_path": reference_path,
        "result_file": result_file,
    }

    print("Questionnaire:", questionnaire_path)
    print("Reference:    ", reference_path)
    print("Output:       ", result_file)

    return run_info

run = prepare_run(cfg)

Questionnaire: C:\Questionnaire_Validation\MMR_R13 household_questionnaire_t1_geopoll_EN_template_20250708_ISO3.xlsx
Reference:     C:\Users\edoar\Food and Agriculture Organization\OER - 01. DIEM Monitoring\03. Toolbox\02. Questionnaires\household_questionnaire_geopoll_EN_template_20250708_ISO3.xlsx
Output:        C:\Questionnaire_Validation\output\MMR_R13_geopoll_validated.xlsx


### step2 normalization ### 

In [35]:
import openpyxl
import polars as pl

import openpyxl
import polars as pl
from pathlib import Path

import openpyxl
import polars as pl
from pathlib import Path

def read_survey_sheet(path: Path, sheet_name: str = "survey", header_row: int = 3) -> pl.DataFrame:
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)

    def get_sheet_case_insensitive(wb, target_name):
        for name in wb.sheetnames:
            if name.strip().lower() == target_name.lower():
                return wb[name]
        raise KeyError(f"Worksheet '{target_name}' not found. Available sheets: {wb.sheetnames}")

    ws = get_sheet_case_insensitive(wb, sheet_name)

    def clean_header(h):
        if h is None:
            return None
        return str(h).replace("\n", " ").strip()

    row_iter = ws.iter_rows(values_only=True)

    for _ in range(header_row - 1):
        next(row_iter)

    raw_headers = next(row_iter)

    headers = [
        clean_header(h) if h is not None else f"unnamed_{i}"
        for i, h in enumerate(raw_headers, start=1)
    ]

    rows = []
    excel_row = header_row + 1

    for values in row_iter:
        if all(v is None for v in values):
            excel_row += 1
            continue

        row_dict = {headers[i]: values[i] for i in range(len(headers))}
        row_dict["excel_row"] = excel_row
        row_dict["source_file"] = path.name
        rows.append(row_dict)

        excel_row += 1

    df = pl.DataFrame(rows)

    if "Q Name" not in df.columns:
        raise ValueError(f"'Q Name' column not found in {path.name}")

    df = df.filter(pl.col("Q Name").is_not_null())

    df = df.with_columns(
        pl.col("Q Name").cast(pl.Utf8).str.strip_chars().alias("Q Name")
    )

    return df

CORE_COLUMNS = [
    "Q Name",
    "Suggested Qname",
    "English",
    "French",
    "Spanish",
    "Arabic",
    "Portuguese",
    "Q Type",
    "Mandatory",
    "Default skip patterns & conditional",
    "Specific skip pattern variable (from blue text)",
    "Additional notes",
    "excel_row",
    "source_file",
]

def build_questions_df(df: pl.DataFrame) -> pl.DataFrame:
    available = [c for c in CORE_COLUMNS if c in df.columns]
    out = df.select(available)

    for c in out.columns:
        if c not in ["excel_row"]:
            out = out.with_columns(
                pl.col(c).cast(pl.Utf8).fill_null("").str.strip_chars().alias(c)
            )

    return out
import re

OPTION_PATTERN = re.compile(r"(?ms)^\s*(\d+)\)\s*(.*?)(?=^\s*\d+\)|\Z)")

def parse_option_blocks(text: str):
    if not text:
        return []
    return [(int(code), label.strip()) for code, label in OPTION_PATTERN.findall(text)]

def explode_options(questions_df: pl.DataFrame, text_col: str = "English") -> pl.DataFrame:
    rows = []

    needed = [c for c in ["Q Name", "Q Type", "excel_row", text_col] if c in questions_df.columns]
    for row in questions_df.select(needed).iter_rows(named=True):
        q_name = row.get("Q Name", "")
        q_type = (row.get("Q Type", "") or "").strip()
        excel_row = row.get("excel_row")
        text = row.get(text_col, "")

        if q_type not in {"Single Choice", "Select All That Apply"}:
            continue

        parsed = parse_option_blocks(text)
        for option_code, option_label in parsed:
            rows.append(
                {
                    "Q Name": q_name,
                    "Q Type": q_type,
                    "option_code": option_code,
                    "option_label": option_label,
                    "text_column": text_col,
                    "excel_row": excel_row,
                }
            )

    if not rows:
        return pl.DataFrame(
            schema={
                "Q Name": pl.Utf8,
                "Q Type": pl.Utf8,
                "option_code": pl.Int64,
                "option_label": pl.Utf8,
                "text_column": pl.Utf8,
                "excel_row": pl.Int64,
            }
        )

    return pl.DataFrame(rows)

In [36]:
current_raw = read_survey_sheet(run["questionnaire_path"])
reference_raw = read_survey_sheet(run["reference_path"])

current_questions = build_questions_df(current_raw)
reference_questions = build_questions_df(reference_raw)

current_options_en = explode_options(current_questions, text_col="English")
reference_options_en = explode_options(reference_questions, text_col="English")

In [ ]:
print(current_questions.shape)
print(reference_questions.shape)
print(current_options_en.shape)
print(current_options_en.head())

(207, 8)
(180, 8)
(1259, 6)
shape: (5, 6)
┌───────────┬───────────────┬─────────────┬────────────────────┬─────────────┬───────────┐
│ Q Name    ┆ Q Type        ┆ option_code ┆ option_label       ┆ text_column ┆ excel_row │
│ ---       ┆ ---           ┆ ---         ┆ ---                ┆ ---         ┆ ---       │
│ str       ┆ str           ┆ i64         ┆ str                ┆ str         ┆ i64       │
╞═══════════╪═══════════════╪═════════════╪════════════════════╪═════════════╪═══════════╡
│ calldispo ┆ Single Choice ┆ 1           ┆ Someone answers    ┆ English     ┆ 6         │
│ calldispo ┆ Single Choice ┆ 2           ┆ No answer          ┆ English     ┆ 6         │
│ calldispo ┆ Single Choice ┆ 3           ┆ Refusal            ┆ English     ┆ 6         │
│ calldispo ┆ Single Choice ┆ 4           ┆ Call back          ┆ English     ┆ 6         │
│ calldispo ┆ Single Choice ┆ 5           ┆ Non-Working Number ┆ English     ┆ 6         │
└───────────┴───────────────┴─────────────┴─────

In [38]:
print(
    current_options_en
    .group_by("Q Type")
    .count()
    .sort("Q Type")
)

shape: (2, 2)
┌───────────────────────┬───────┐
│ Q Type                ┆ count │
│ ---                   ┆ ---   │
│ str                   ┆ u32   │
╞═══════════════════════╪═══════╡
│ Select All That Apply ┆ 114   │
│ Single Choice         ┆ 1145  │
└───────────────────────┴───────┘


C:\Users\edoar\AppData\Local\Temp\ipykernel_20116\1718789111.py:4: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


In [39]:
option_qtypes = {"Single Choice", "Select All That Apply"}

eligible_current = (
    current_questions
    .with_columns(pl.col("Q Type").str.strip_chars().alias("Q Type"))
    .filter(pl.col("Q Type").is_in(option_qtypes))
    .select("Q Name")
    .unique()
)

parsed_current = (
    current_options_en
    .select("Q Name")
    .unique()
)

print("Eligible questions:", eligible_current.height)
print("Parsed questions:  ", parsed_current.height)
print("Missing after explosion:")
print(
    eligible_current
    .join(parsed_current, on="Q Name", how="anti")
    .sort("Q Name")
)

Eligible questions: 144
Parsed questions:   144
Missing after explosion:
shape: (0, 1)
┌────────┐
│ Q Name │
│ ---    │
│ str    │
╞════════╡
└────────┘
